# demo_rl17

> Phase 17 (`specs/multiagent/phase17/`): RL that makes larger teams pay off. On the Mac, from the Colab outputs: step 0's baseline curves (every existing adapter at 4-32 agents), the pilot's selection and the projected main-run hours (for the user to confirm before `RL17_PLAN` is frozen), and the main analysis (`RL17_PLAN`'s hypotheses, curves against Claude Haiku's references).

Artifacts (gitignored): `results/multiagent/rl/{quick_p17, eval_p17s0, pilot_p17, eval_p17}/`. Inputs: the Colab zips unzipped into `results/multiagent/rl/colab/` (`eval_p17s0/`, `runs_p17pilot/`, `eval_p17/`); Haiku references from `results/multiagent/{clue,scale,fix}/full/runs.csv`.

- **Quick** (normal tests): synthetic runs end to end.
- **Step 0, pilot, main** (`--flags train`): each only when its Colab output is present.

In [ ]:
import json
import subprocess
from pathlib import Path

import numpy as np

from explRL.bench_grid import readCSV
from explRL.coop_grid import coopHash
from explRL.rl17_grid import PILOT17_PLAN, RL17_PLAN, SIZES, STEP0_MODELS, chosenRecipe, pilotChoice, projectHours
from explRL.rl17_report import curves, figRL17, haikuRefs, rl17Summaries, syntheticRows17, writeRL17
from explRL.train import demo_artifact_dir

RL = demo_artifact_dir(None, 'multiagent/rl')
MA, COLAB = RL.parent, RL / 'colab'


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def refs():
    p = [MA / f'{x}/full/runs.csv' for x in ('clue', 'scale', 'fix')]
    return haikuRefs(*map(readCSV, p)) if all(x.exists() for x in p) else {}


def analyse(mode, rows, out, s0rows=None, nboot=2000, rf=None):
    out.mkdir(parents=True, exist_ok=True)
    S = rl17Summaries(rows, s0rows, rf, nboot)
    meta = dict(mode=mode, planHash=coopHash(RL17_PLAN), commit=gitCommit())
    writeRL17(out / 'report.md', S, meta)
    figRL17(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, **S), indent=1, sort_keys=True, default=str) + '\n')
    return S

## Quick: synthetic runs

In [ ]:
qo = RL / 'quick_p17'
S = analyse('quick', syntheticRows17(), qo, nboot=200, rf={N: dict(welfare=0.5, pooledShare=0.4, honestLoss=0.5) for N in SIZES})
assert all((qo / f).stat().st_size > 0 for f in ('report.md', 'rl17.png', 'summary.json'))
assert all(S['hyp'][k]['supported'] for k in S['hyp'])
print(open(qo / 'report.md').read()[:1500])

## Step 0 (`--flags train`): the baseline curves of the existing adapters

In [ ]:
#| train
p0 = COLAB / 'eval_p17s0' / 'runs.csv'
if not p0.exists():
    print('skipped: unzip explRL_rl17s0.zip into results/multiagent/rl/colab/ first')
else:
    C = curves(readCSV(p0), list(STEP0_MODELS))
    rf = refs()
    o = RL / 'eval_p17s0'
    o.mkdir(parents=True, exist_ok=True)
    L = ['# Phase 17 step 0: existing adapters against team size (descriptive)', '']
    for k in ('welfare', 'pooledShare', 'honestLoss', 'sharePerHeld', 'invalid'):
        L += [f'## {k}', '', '| model | ' + ' | '.join(f'N {N}' for N in SIZES) + ' |', '|---|' + '---|' * len(SIZES)]
        L += [f'| {m} | ' + ' | '.join(f'{C[m][N][k]:.3f}' for N in SIZES) + ' |' for m in C]
        if rf and k in ('welfare', 'pooledShare', 'honestLoss'):
            L.append('| Claude Haiku (reference) | ' + ' | '.join(f'{rf[N][k]:.3f}' for N in SIZES) + ' |')
        L.append('')
    (o / 'report.md').write_text('\n'.join(L) + '\n')
    (o / 'curves.json').write_text(json.dumps(C, indent=1, sort_keys=True) + '\n')
    print((o / 'report.md').read_text())

## Pilot (`--flags train`): the selection and the projected main-run hours

The rule fixed in `PILOT17_PLAN` picks the recipe; the projection uses the chosen run's measured seconds per step. The main runs start only after the user confirms these hours; then `RL17_PLAN['recipe']`, `['projectedHours']` and `['userConfirmed']` are set and the plan is frozen.

In [ ]:
#| train
P = COLAB / PILOT17_PLAN['dir']
if not (P / 'selection.json').exists():
    print(f"skipped: unzip {PILOT17_PLAN['zip']} into results/multiagent/rl/colab/ first")
else:
    assert PILOT17_PLAN['frozen']
    names = [n for n in ('p17a', 'p17b', 'p17c', 'p17d') if (P / n / 'metrics.jsonl').exists()]
    met = {n: [json.loads(x) for x in open(P / n / 'metrics.jsonl')] for n in names}
    choice, table = pilotChoice(met, {n for n in names if (P / n / 'halted.json').exists()}, names=names)
    assert choice == json.loads((P / 'selection.json').read_text())['choice'], 'the Mac and Colab disagree on the choice'
    sel = json.loads((P / 'selection.json').read_text())
    recipe = chosenRecipe(choice, sel.get('p17dBase')) if choice else None
    h = projectHours(table[choice]['seconds']) if choice else None
    o = RL / 'pilot_p17'
    o.mkdir(parents=True, exist_ok=True)
    (o / 'selection.json').write_text(json.dumps(dict(choice=choice, recipe=recipe, table=table, projectedHours=h), indent=1, default=str) + '\n')
    for n, t in table.items():
        print(n, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in t.items()})
    print('choice:', choice, '; recipe for RL17_PLAN:', recipe, '; projected main-run A100 hours:', h)

## Main (`--flags train`): `RL17_PLAN`'s analysis

In [ ]:
#| train
p17, p0 = COLAB / 'eval_p17' / 'runs.csv', COLAB / 'eval_p17s0' / 'runs.csv'
if not p17.exists():
    print('skipped: unzip explRL_rl17.zip into results/multiagent/rl/colab/ first')
else:
    assert RL17_PLAN['frozen'], 'the analysis uses the frozen RL17_PLAN'
    S = analyse('full', readCSV(p17), RL / 'eval_p17', readCSV(p0), rf=refs())
    print(open(RL / 'eval_p17' / 'report.md').read())